# SPY 5-Minute Signal Arrows + ML + Backtest

A clean SPY 5-minute candlestick chart with **only buy/sell arrows** on it, driven
by a rule engine (EMA/MACD/RSI/VWAP/Bollinger, clustered support/resistance,
sloped trend lines, classic chart patterns, harmonic patterns) **combined with a
machine-learning model** (RandomForest) trained to predict the next ~30 minutes
of price movement from the same engineered features.

Everything is computed **causally, bar by bar** — at every point in history, only
information available up to that bar is used. The ML model is trained only on the
**older ~70% of the fetched history** and evaluated / backtested only on the
**newer ~30%** it never saw during training, so the accuracy numbers you see are
genuinely out-of-sample, not curve-fit.

**How to use in Google Colab**
1. Run the *Install packages* cell once per session.
2. Run every cell top to bottom once. The last cell fetches ~60 days of 5-minute
   bars (Yahoo's max for this interval), trains the model, prints a backtest
   report over the held-out period, and plots today's candlestick chart with
   arrows.
3. Re-run just the **last cell** (`run_pipeline()`) as often as you like during
   the trading day for an updated signal — it refits the model on the latest data
   each time.
4. Or run the *Live loop* cell to have it auto-refresh every few minutes on its own.

**Disclaimer:** Educational tool only, not financial advice. Free `yfinance` data
can lag real-time by up to ~15 minutes, and a backtest — even an honest
out-of-sample one — is not a guarantee of future performance. Transaction costs,
slippage, and the fact that SPY sell signals are backtested as if shorting are
all simplified away here. Validate carefully before trading real money.


## 1. Install packages

In [ ]:
!pip install -q yfinance plotly scipy scikit-learn


## 2. Imports & configuration

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import yfinance as yf
from scipy.signal import argrelextrema
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import plotly.graph_objects as go

# ---- Configuration ----
SYMBOL = "SPY"
INTERVAL = "5m"        # 5-minute candles
PERIOD = "60d"         # Yahoo's max lookback for 5m bars — gives a wide backtest range
PIVOT_WINDOW = 5              # bars each side required to confirm a swing high/low
SR_TOLERANCE_PCT = 0.0015     # cluster tolerance for horizontal support/resistance (0.15%)
HARMONIC_TOLERANCE = 0.07     # tolerance band around ideal Fibonacci ratios
TRENDLINE_LOOKBACK = 6        # how many recent swing points to fit each trend line on
CAUSAL_LOOKBACK = 150         # bars of history the rule engine looks back over at each step
WARMUP_BARS = 60              # bars needed before the first signal can be computed

# ---- Machine learning configuration ----
FORWARD_HORIZON = 6           # bars ahead the model predicts (6 x 5min = 30 min)
LABEL_THRESHOLD = 0.0006      # forward move must exceed this (0.06%) to count as up/down, else "flat"
TRAIN_FRACTION = 0.7          # older 70% of history -> training, newer 30% -> honest out-of-sample test
ML_WEIGHT = 1.0               # how much the ML model's opinion counts vs. the rule engine's
RANDOM_STATE = 42

FEATURE_COLS = [
    "ema_stack", "ema9_21_pct", "ema21_50_pct", "macd_bias", "macd_hist_pct", "rsi",
    "vwap_diff_pct", "bb_pct", "atr_pct", "vol_ratio", "dist_support_pct", "dist_resistance_pct",
    "resistance_break", "support_break", "classic_bull", "classic_bear", "harmonic_bull",
    "harmonic_bear", "rule_score",
]


## 3. Data fetch

In [ ]:
def fetch_data(symbol=SYMBOL, interval=INTERVAL, period=PERIOD):
    df = yf.download(symbol, interval=interval, period=period, progress=False, auto_adjust=False)
    if df.empty:
        raise ValueError("No data returned — market may be closed, symbol invalid, or rate-limited.")
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df.index.name = "Datetime"
    if df.index.tz is None:
        df = df.tz_localize("UTC")
    df = df.tz_convert("America/New_York")
    df = df[["Open", "High", "Low", "Close", "Volume"]].dropna()
    return df


## 4. Indicators (EMA, MACD, RSI, Bollinger Bands, ATR, VWAP, Volume SMA)

In [ ]:
def add_indicators(df):
    df = df.copy()

    df["EMA9"] = df["Close"].ewm(span=9, adjust=False).mean()
    df["EMA21"] = df["Close"].ewm(span=21, adjust=False).mean()
    df["EMA50"] = df["Close"].ewm(span=50, adjust=False).mean()

    ema12 = df["Close"].ewm(span=12, adjust=False).mean()
    ema26 = df["Close"].ewm(span=26, adjust=False).mean()
    df["MACD"] = ema12 - ema26
    df["MACD_signal"] = df["MACD"].ewm(span=9, adjust=False).mean()
    df["MACD_hist"] = df["MACD"] - df["MACD_signal"]

    delta = df["Close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / 14, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / 14, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    df["RSI"] = 100 - (100 / (1 + rs))
    df["RSI"] = df["RSI"].fillna(50)

    sma20 = df["Close"].rolling(20).mean()
    std20 = df["Close"].rolling(20).std()
    df["BB_mid"] = sma20
    df["BB_upper"] = sma20 + 2 * std20
    df["BB_lower"] = sma20 - 2 * std20

    high_low = df["High"] - df["Low"]
    high_close = (df["High"] - df["Close"].shift()).abs()
    low_close = (df["Low"] - df["Close"].shift()).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    df["ATR"] = tr.ewm(alpha=1 / 14, adjust=False).mean()

    df["VolSMA20"] = df["Volume"].rolling(20).mean()

    # Session VWAP — resets every trading day. All of the above are rolling/ewm
    # (backward-looking only) so every value here is causal: it only uses bars
    # up to and including its own row.
    session_date = df.index.date
    typical = (df["High"] + df["Low"] + df["Close"]) / 3
    tpv = typical * df["Volume"]
    df["VWAP"] = tpv.groupby(session_date).cumsum() / df["Volume"].groupby(session_date).cumsum()

    return df


## 5. Swing pivots, support/resistance clusters, trend lines

In [ ]:
def find_pivots(df, window=PIVOT_WINDOW):
    highs = df["High"].values
    lows = df["Low"].values
    piv_high_idx = argrelextrema(highs, np.greater, order=window)[0]
    piv_low_idx = argrelextrema(lows, np.less, order=window)[0]
    return sorted(piv_high_idx.tolist()), sorted(piv_low_idx.tolist())


def cluster_levels(df, piv_idx, price_col, tolerance_pct=SR_TOLERANCE_PCT, min_touches=2, max_levels=6):
    prices = df[price_col].values[piv_idx]
    if len(prices) == 0:
        return []
    levels = []
    for p in sorted(prices):
        placed = False
        for lvl in levels:
            if abs(p - lvl["price"]) / lvl["price"] <= tolerance_pct:
                lvl["prices"].append(p)
                lvl["price"] = float(np.mean(lvl["prices"]))
                lvl["touches"] += 1
                placed = True
                break
        if not placed:
            levels.append({"price": float(p), "prices": [p], "touches": 1})
    levels = [lvl for lvl in levels if lvl["touches"] >= min_touches]
    levels.sort(key=lambda x: -x["touches"])
    return levels[:max_levels]


def get_support_resistance(df, piv_high_idx, piv_low_idx):
    resistance = cluster_levels(df, piv_high_idx, "High")
    support = cluster_levels(df, piv_low_idx, "Low")
    return support, resistance


def fit_trendline(idx_list, price_list, n_recent=TRENDLINE_LOOKBACK):
    if len(idx_list) < 2:
        return None
    idx_arr = np.array(idx_list[-n_recent:], dtype=float)
    price_arr = np.array(price_list[-n_recent:], dtype=float)
    slope, intercept = np.polyfit(idx_arr, price_arr, 1)
    return float(slope), float(intercept)


def get_trendlines(df, piv_high_idx, piv_low_idx):
    resistance_line = None
    support_line = None
    if len(piv_high_idx) >= 2:
        resistance_line = fit_trendline(piv_high_idx, df["High"].values[piv_high_idx])
    if len(piv_low_idx) >= 2:
        support_line = fit_trendline(piv_low_idx, df["Low"].values[piv_low_idx])
    return support_line, resistance_line


## 6. Classic chart patterns (double top/bottom, head & shoulders, triangles)

In [ ]:
def detect_double_top_bottom(df, piv_high_idx, piv_low_idx, tolerance=0.002):
    patterns = []
    if len(piv_high_idx) >= 2:
        i1, i2 = piv_high_idx[-2], piv_high_idx[-1]
        p1, p2 = df["High"].iloc[i1], df["High"].iloc[i2]
        if abs(p1 - p2) / p1 <= tolerance:
            neckline = df["Low"].iloc[i1:i2 + 1].min()
            confirmed = bool(df["Close"].iloc[-1] < neckline)
            patterns.append({"pattern": "Double Top", "bias": "bearish",
                              "neckline": float(neckline), "index": i2, "confirmed": confirmed})
    if len(piv_low_idx) >= 2:
        i1, i2 = piv_low_idx[-2], piv_low_idx[-1]
        p1, p2 = df["Low"].iloc[i1], df["Low"].iloc[i2]
        if abs(p1 - p2) / p1 <= tolerance:
            neckline = df["High"].iloc[i1:i2 + 1].max()
            confirmed = bool(df["Close"].iloc[-1] > neckline)
            patterns.append({"pattern": "Double Bottom", "bias": "bullish",
                              "neckline": float(neckline), "index": i2, "confirmed": confirmed})
    return patterns


def detect_head_shoulders(df, piv_high_idx, piv_low_idx, tolerance=0.01):
    patterns = []
    if len(piv_high_idx) >= 3:
        i1, i2, i3 = piv_high_idx[-3:]
        p1, p2, p3 = df["High"].iloc[i1], df["High"].iloc[i2], df["High"].iloc[i3]
        if p2 > p1 and p2 > p3 and abs(p1 - p3) / p1 <= tolerance:
            neckline = df["Low"].iloc[i1:i3 + 1].min()
            confirmed = bool(df["Close"].iloc[-1] < neckline)
            patterns.append({"pattern": "Head & Shoulders", "bias": "bearish",
                              "neckline": float(neckline), "index": i3, "confirmed": confirmed})
    if len(piv_low_idx) >= 3:
        i1, i2, i3 = piv_low_idx[-3:]
        p1, p2, p3 = df["Low"].iloc[i1], df["Low"].iloc[i2], df["Low"].iloc[i3]
        if p2 < p1 and p2 < p3 and abs(p1 - p3) / p1 <= tolerance:
            neckline = df["High"].iloc[i1:i3 + 1].max()
            confirmed = bool(df["Close"].iloc[-1] > neckline)
            patterns.append({"pattern": "Inverse Head & Shoulders", "bias": "bullish",
                              "neckline": float(neckline), "index": i3, "confirmed": confirmed})
    return patterns


def detect_triangle(support_line, resistance_line, avg_price, flat_thresh_pct=0.0002):
    if support_line is None or resistance_line is None or avg_price == 0:
        return None
    s_slope_pct = support_line[0] / avg_price
    r_slope_pct = resistance_line[0] / avg_price
    if abs(r_slope_pct) <= flat_thresh_pct and s_slope_pct > flat_thresh_pct:
        return {"pattern": "Ascending Triangle", "bias": "bullish", "confirmed": False}
    if abs(s_slope_pct) <= flat_thresh_pct and r_slope_pct < -flat_thresh_pct:
        return {"pattern": "Descending Triangle", "bias": "bearish", "confirmed": False}
    if r_slope_pct < -flat_thresh_pct and s_slope_pct > flat_thresh_pct:
        return {"pattern": "Symmetrical Triangle", "bias": "neutral", "confirmed": False}
    return None


## 7. Harmonic patterns (Gartley, Bat, Butterfly, Crab)

In [ ]:
HARMONIC_DEFS = {
    "Gartley":   {"AB_XA": (0.618, 0.618), "BC_AB": (0.382, 0.886), "CD_BC": (1.13, 1.618), "AD_XA": (0.786, 0.786)},
    "Bat":       {"AB_XA": (0.382, 0.5),   "BC_AB": (0.382, 0.886), "CD_BC": (1.618, 2.618), "AD_XA": (0.886, 0.886)},
    "Butterfly": {"AB_XA": (0.786, 0.786), "BC_AB": (0.382, 0.886), "CD_BC": (1.618, 2.24),  "AD_XA": (1.27, 1.618)},
    "Crab":      {"AB_XA": (0.382, 0.618), "BC_AB": (0.382, 0.886), "CD_BC": (2.24, 3.618),  "AD_XA": (1.618, 1.618)},
}


def get_alternating_pivots(df, piv_high_idx, piv_low_idx):
    """Merge swing highs/lows into one time-ordered, strictly alternating sequence."""
    points = [(i, float(df["High"].iloc[i]), "H") for i in piv_high_idx]
    points += [(i, float(df["Low"].iloc[i]), "L") for i in piv_low_idx]
    points.sort(key=lambda x: x[0])
    filtered = []
    for pt in points:
        if not filtered:
            filtered.append(pt)
            continue
        last = filtered[-1]
        if pt[2] == last[2]:
            if pt[2] == "H" and pt[1] > last[1]:
                filtered[-1] = pt
            elif pt[2] == "L" and pt[1] < last[1]:
                filtered[-1] = pt
        else:
            filtered.append(pt)
    return filtered


def ratio_in_range(value, lo, hi, tol=HARMONIC_TOLERANCE):
    return (lo - tol) <= value <= (hi + tol)


def detect_harmonic_patterns(alt_points):
    results = []
    if len(alt_points) < 5:
        return results
    X, A, B, C, D = alt_points[-5:]
    xi, xp, xt = X
    ai, ap, at = A
    bi, bp, bt = B
    ci, cp, ct = C
    di, dp, dt = D

    if not (xt == bt == dt and at == ct and xt != at):
        return results

    bullish = xt == "L"  # X,B,D are swing lows -> pattern completes into a bullish reversal at D
    XA, AB, BC, CD, AD = abs(ap - xp), abs(bp - ap), abs(cp - bp), abs(dp - cp), abs(dp - ap)
    if XA == 0 or AB == 0 or BC == 0:
        return results

    ab_xa, bc_ab, cd_bc, ad_xa = AB / XA, BC / AB, CD / BC, AD / XA
    for name, r in HARMONIC_DEFS.items():
        if (ratio_in_range(ab_xa, *r["AB_XA"]) and ratio_in_range(bc_ab, *r["BC_AB"]) and
                ratio_in_range(cd_bc, *r["CD_BC"]) and ratio_in_range(ad_xa, *r["AD_XA"])):
            results.append({
                "pattern": name,
                "bias": "bullish" if bullish else "bearish",
                "D_index": di,
                "D_price": dp,
            })
    return results


## 8. Rule-based signal scoring + feature engineering for ML

In [ ]:
def generate_signal(df, support, resistance, support_line, resistance_line,
                     classic_patterns, harmonic_patterns):
    """Scores the LAST bar of `df` and also returns a dict of numeric features
    describing that same bar — the features feed the ML model below, while the
    score/reasons remain a fully human-readable rule-based opinion on their own."""
    last = df.iloc[-1]
    n = len(df) - 1
    score = 0
    reasons = []
    feat = {}

    ema_stack = 0
    if last["EMA9"] > last["EMA21"] > last["EMA50"]:
        score += 2; reasons.append("EMA9 > EMA21 > EMA50 (uptrend)"); ema_stack = 1
    elif last["EMA9"] < last["EMA21"] < last["EMA50"]:
        score -= 2; reasons.append("EMA9 < EMA21 < EMA50 (downtrend)"); ema_stack = -1
    feat["ema_stack"] = ema_stack
    feat["ema9_21_pct"] = (last["EMA9"] - last["EMA21"]) / last["Close"]
    feat["ema21_50_pct"] = (last["EMA21"] - last["EMA50"]) / last["Close"]

    macd_bias = 0
    if last["MACD"] > last["MACD_signal"] and last["MACD_hist"] > 0:
        score += 1; reasons.append("MACD bullish crossover"); macd_bias = 1
    elif last["MACD"] < last["MACD_signal"] and last["MACD_hist"] < 0:
        score -= 1; reasons.append("MACD bearish crossover"); macd_bias = -1
    feat["macd_bias"] = macd_bias
    feat["macd_hist_pct"] = last["MACD_hist"] / last["Close"]

    if last["RSI"] < 30:
        score += 1; reasons.append(f"RSI oversold ({last['RSI']:.1f})")
    elif last["RSI"] > 70:
        score -= 1; reasons.append(f"RSI overbought ({last['RSI']:.1f})")
    feat["rsi"] = last["RSI"]

    if last["Close"] > last["VWAP"]:
        score += 1; reasons.append("Price above VWAP")
    else:
        score -= 1; reasons.append("Price below VWAP")
    feat["vwap_diff_pct"] = (last["Close"] - last["VWAP"]) / last["VWAP"]

    bb_range = last["BB_upper"] - last["BB_lower"]
    feat["bb_pct"] = (last["Close"] - last["BB_mid"]) / bb_range if bb_range > 0 else 0.0
    if last["Close"] <= last["BB_lower"]:
        score += 1; reasons.append("Price at/below lower Bollinger Band")
    elif last["Close"] >= last["BB_upper"]:
        score -= 1; reasons.append("Price at/above upper Bollinger Band")

    feat["atr_pct"] = last["ATR"] / last["Close"]
    feat["vol_ratio"] = (last["Volume"] / last["VolSMA20"]) if last["VolSMA20"] > 0 else 1.0

    support_dists = [abs(last["Close"] - lvl["price"]) / lvl["price"] for lvl in support]
    resistance_dists = [abs(last["Close"] - lvl["price"]) / lvl["price"] for lvl in resistance]
    feat["dist_support_pct"] = min(support_dists) if support_dists else np.nan
    feat["dist_resistance_pct"] = min(resistance_dists) if resistance_dists else np.nan
    for lvl in support:
        if abs(last["Close"] - lvl["price"]) / lvl["price"] <= 0.002:
            score += 1; reasons.append(f"Near support {lvl['price']:.2f}")
    for lvl in resistance:
        if abs(last["Close"] - lvl["price"]) / lvl["price"] <= 0.002:
            score -= 1; reasons.append(f"Near resistance {lvl['price']:.2f}")

    resistance_break = 0
    if resistance_line is not None:
        r_val = resistance_line[0] * n + resistance_line[1]
        if last["Close"] > r_val:
            score += 2; reasons.append("Breakout above resistance trend line"); resistance_break = 1
    feat["resistance_break"] = resistance_break

    support_break = 0
    if support_line is not None:
        s_val = support_line[0] * n + support_line[1]
        if last["Close"] < s_val:
            score -= 2; reasons.append("Breakdown below support trend line"); support_break = 1
    feat["support_break"] = support_break

    classic_bull = 0
    classic_bear = 0
    for pat in classic_patterns:
        if pat.get("confirmed"):
            if pat["bias"] == "bullish":
                score += 2; reasons.append(f"{pat['pattern']} confirmed (bullish)"); classic_bull = 1
            elif pat["bias"] == "bearish":
                score -= 2; reasons.append(f"{pat['pattern']} confirmed (bearish)"); classic_bear = 1
    feat["classic_bull"] = classic_bull
    feat["classic_bear"] = classic_bear

    harmonic_bull = 0
    harmonic_bear = 0
    for pat in harmonic_patterns:
        if abs(pat["D_index"] - n) <= 3:
            if pat["bias"] == "bullish":
                score += 3; reasons.append(f"{pat['pattern']} harmonic bullish completion at D"); harmonic_bull = 1
            else:
                score -= 3; reasons.append(f"{pat['pattern']} harmonic bearish completion at D"); harmonic_bear = 1
    feat["harmonic_bull"] = harmonic_bull
    feat["harmonic_bear"] = harmonic_bear
    feat["rule_score"] = score

    if score >= 4:
        signal = "STRONG BUY"
    elif score >= 2:
        signal = "BUY"
    elif score <= -4:
        signal = "STRONG SELL"
    elif score <= -2:
        signal = "SELL"
    else:
        signal = "HOLD"

    return signal, score, reasons, feat


## 9. Causal, bar-by-bar signal + feature history

In [ ]:
def compute_signal_history(df, lookback=CAUSAL_LOOKBACK, warmup=WARMUP_BARS):
    """Walks forward bar by bar. At bar i, only uses rows [i-lookback+1, i] — so a
    signal (and its features) at bar i never sees data from bar i+1 onward. This is
    what makes both the rule engine and the ML training set free of lookahead bias."""
    n = len(df)
    signals = ["HOLD"] * n
    scores = [0] * n
    reasons_list = [[] for _ in range(n)]
    feats = [{k: np.nan for k in FEATURE_COLS} for _ in range(n)]

    for i in range(warmup, n):
        start = max(0, i - lookback + 1)
        window = df.iloc[start:i + 1]

        piv_high_idx, piv_low_idx = find_pivots(window)
        support, resistance = get_support_resistance(window, piv_high_idx, piv_low_idx)
        support_line, resistance_line = get_trendlines(window, piv_high_idx, piv_low_idx)

        classic_patterns = []
        classic_patterns += detect_double_top_bottom(window, piv_high_idx, piv_low_idx)
        classic_patterns += detect_head_shoulders(window, piv_high_idx, piv_low_idx)
        triangle = detect_triangle(support_line, resistance_line, window["Close"].mean())
        if triangle:
            classic_patterns.append(triangle)

        alt_points = get_alternating_pivots(window, piv_high_idx, piv_low_idx)
        harmonic_patterns = detect_harmonic_patterns(alt_points)

        signal, score, reasons, feat = generate_signal(window, support, resistance, support_line,
                                                         resistance_line, classic_patterns, harmonic_patterns)
        signals[i] = signal
        scores[i] = score
        reasons_list[i] = reasons
        feats[i] = feat

    out = df.copy()
    out["Signal"] = signals
    out["Score"] = scores
    out["Reasons"] = reasons_list
    feat_df = pd.DataFrame(feats, index=df.index)
    out = pd.concat([out, feat_df], axis=1)
    return out


## 10. Machine learning — label, split, train, predict

In [ ]:
def add_forward_labels(df, horizon=FORWARD_HORIZON, threshold=LABEL_THRESHOLD):
    """Label = 1 if price is up more than `threshold` after `horizon` bars, -1 if
    down more than `threshold`, else 0 (flat/no edge). The last `horizon` bars of
    the whole dataset can't be labeled (no future data yet) and are left NaN."""
    df = df.copy()
    fwd_ret = df["Close"].shift(-horizon) / df["Close"] - 1
    label = pd.Series(0.0, index=df.index)
    label[fwd_ret > threshold] = 1
    label[fwd_ret < -threshold] = -1
    label[fwd_ret.isna()] = np.nan
    df["FwdReturn"] = fwd_ret
    df["Label"] = label
    return df


def chronological_split(df, feature_cols, train_fraction=TRAIN_FRACTION):
    """Time-ordered split — training uses only the OLDER portion, testing only the
    NEWER portion, so no future information ever leaks into training."""
    usable = df.dropna(subset=feature_cols + ["Label"])
    split_i = int(len(usable) * train_fraction)
    train_idx = usable.index[:split_i]
    test_idx = usable.index[split_i:]
    return train_idx, test_idx


def prep_features(df, feature_cols=FEATURE_COLS):
    X = df[feature_cols].copy()
    # NaN distance-to-level means "no support/resistance level was found nearby" —
    # encode that as simply "far away" rather than leaving it as NaN for sklearn.
    X["dist_support_pct"] = X["dist_support_pct"].fillna(0.05)
    X["dist_resistance_pct"] = X["dist_resistance_pct"].fillna(0.05)
    return X.fillna(0.0)


def train_ml_model(df, train_idx, feature_cols=FEATURE_COLS, random_state=RANDOM_STATE):
    X_train = prep_features(df.loc[train_idx], feature_cols).values
    y_train = df.loc[train_idx, "Label"].values
    model = RandomForestClassifier(
        n_estimators=300, max_depth=6, min_samples_leaf=20,
        class_weight="balanced", random_state=random_state, n_jobs=-1,
    )
    model.fit(X_train, y_train)
    return model


def add_ml_predictions(df, model, feature_cols=FEATURE_COLS):
    df = df.copy()
    mask = df[feature_cols].notna().all(axis=1)
    proba = model.predict_proba(prep_features(df.loc[mask], feature_cols).values)
    classes = list(model.classes_)
    df["P_up"] = np.nan
    df["P_down"] = np.nan
    df["P_flat"] = np.nan
    if 1 in classes:
        df.loc[mask, "P_up"] = proba[:, classes.index(1)]
    if -1 in classes:
        df.loc[mask, "P_down"] = proba[:, classes.index(-1)]
    if 0 in classes:
        df.loc[mask, "P_flat"] = proba[:, classes.index(0)]
    return df


def combine_rule_and_ml(df, ml_weight=ML_WEIGHT):
    """Final score = rule-engine score, nudged by how strongly the ML model leans
    up vs. down. A rule-only BUY that the model actively disagrees with gets
    pulled back toward HOLD instead of firing — this is what should make signals
    more selective/accurate than the rule engine alone."""
    df = df.copy()
    ml_component = (df["P_up"].fillna(0) - df["P_down"].fillna(0)) * 10
    df["FinalScore"] = df["Score"].fillna(0) + ml_weight * ml_component

    def classify(s):
        if pd.isna(s):
            return "HOLD"
        if s >= 4:
            return "STRONG BUY"
        if s >= 2:
            return "BUY"
        if s <= -4:
            return "STRONG SELL"
        if s <= -2:
            return "SELL"
        return "HOLD"

    df["FinalSignal"] = df["FinalScore"].apply(classify)
    return df


def add_arrow_markers(df, signal_col="FinalSignal"):
    """Marks only the bar where a signal *first* turns BUY or SELL, so arrows show
    up once per move instead of on every bar the condition happens to hold."""
    df = df.copy()
    buy_states = {"BUY", "STRONG BUY"}
    sell_states = {"SELL", "STRONG SELL"}
    prev_signal = df[signal_col].shift(1).fillna("HOLD")
    df["BuyArrow"] = df[signal_col].isin(buy_states) & ~prev_signal.isin(buy_states)
    df["SellArrow"] = df[signal_col].isin(sell_states) & ~prev_signal.isin(sell_states)
    return df


## 11. Backtest — simulate every signal on the held-out (out-of-sample) period

In [ ]:
def backtest_signals(df, test_idx, horizon=FORWARD_HORIZON, signal_col="FinalSignal"):
    """For every fresh BUY/SELL arrow in the test period, simulate entering at
    that bar's close and exiting `horizon` bars later (matching what the ML label
    predicts). SELL trades are scored as if shorting — a simplification, since
    SPY shorting in practice usually means an inverse ETF or options."""
    test_df = df.loc[test_idx]
    buy_states = {"BUY", "STRONG BUY"}
    sell_states = {"SELL", "STRONG SELL"}
    prev_signal = test_df[signal_col].shift(1).fillna("HOLD")
    is_entry = (test_df[signal_col].isin(buy_states) & ~prev_signal.isin(buy_states)) | \
               (test_df[signal_col].isin(sell_states) & ~prev_signal.isin(sell_states))
    entries = test_df[is_entry]

    trades = []
    n = len(df)
    for ts, row in entries.iterrows():
        i = df.index.get_loc(ts)
        exit_i = min(i + horizon, n - 1)
        if exit_i <= i:
            continue
        entry_price = df["Close"].iloc[i]
        exit_price = df["Close"].iloc[exit_i]
        direction = "LONG" if row[signal_col] in buy_states else "SHORT"
        ret = (exit_price / entry_price - 1) if direction == "LONG" else (entry_price / exit_price - 1)
        trades.append({"entry_time": ts, "exit_time": df.index[exit_i], "direction": direction,
                        "entry_price": entry_price, "exit_price": exit_price, "return_pct": ret * 100})
    return pd.DataFrame(trades)


def summarize_backtest(trades):
    if trades is None or trades.empty:
        return {"n_trades": 0}
    wins = trades[trades["return_pct"] > 0]
    losses = trades[trades["return_pct"] <= 0]
    gross_win = wins["return_pct"].sum()
    gross_loss = abs(losses["return_pct"].sum())
    equity_curve = (1 + trades["return_pct"] / 100).cumprod()
    running_max = equity_curve.cummax()
    drawdown = (equity_curve - running_max) / running_max
    return {
        "n_trades": len(trades),
        "win_rate_pct": len(wins) / len(trades) * 100,
        "avg_return_pct": trades["return_pct"].mean(),
        "total_return_pct": trades["return_pct"].sum(),
        "profit_factor": (gross_win / gross_loss) if gross_loss > 0 else float("inf"),
        "max_drawdown_pct": drawdown.min() * 100,
        "equity_curve": equity_curve,
    }


def print_backtest_report(trades, stats, label="Backtest"):
    print("-" * 64)
    print(f"{label} — {stats.get('n_trades', 0)} trades")
    if stats.get("n_trades", 0) == 0:
        print("No trades were triggered in this period.")
        print("-" * 64)
        return
    print(f"  Win rate:        {stats['win_rate_pct']:.1f}%")
    print(f"  Avg return/trade:{stats['avg_return_pct']:+.3f}%")
    print(f"  Total return:    {stats['total_return_pct']:+.2f}%  (sum of per-trade returns)")
    print(f"  Profit factor:   {stats['profit_factor']:.2f}  (gross wins / gross losses)")
    print(f"  Max drawdown:    {stats['max_drawdown_pct']:.2f}%  (on the trade equity curve)")
    print("-" * 64)


def plot_equity_curve(stats, symbol=SYMBOL):
    if stats.get("n_trades", 0) == 0:
        return
    curve = stats["equity_curve"]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=list(range(1, len(curve) + 1)), y=curve.values,
                              mode="lines", line=dict(color="#00e676", width=2), name="Equity"))
    fig.update_layout(
        title=f"{symbol} — Out-of-Sample Backtest Equity Curve ({stats['n_trades']} trades)",
        xaxis_title="Trade #", yaxis_title="Equity (starting at 1.0)",
        template="plotly_dark", height=350,
    )
    fig.show()


## 12. Chart — candlesticks with buy/sell arrows only

In [ ]:
def plot_signals_chart(df, symbol=SYMBOL):
    fig = go.Figure()

    fig.add_trace(go.Candlestick(
        x=df.index, open=df["Open"], high=df["High"], low=df["Low"], close=df["Close"],
        name=symbol, increasing_line_color="#26a69a", decreasing_line_color="#ef5350",
    ))

    buys = df[df["BuyArrow"]]
    sells = df[df["SellArrow"]]

    if len(buys):
        fig.add_trace(go.Scatter(
            x=buys.index, y=buys["Low"] * 0.998, mode="markers", name="BUY",
            marker=dict(symbol="triangle-up", size=16, color="#00e676", line=dict(width=1, color="black")),
            text=[f"BUY  score {s:.1f}<br>" + "<br>".join(r) for s, r in zip(buys["FinalScore"], buys["Reasons"])],
            hoverinfo="text+x",
        ))
    if len(sells):
        fig.add_trace(go.Scatter(
            x=sells.index, y=sells["High"] * 1.002, mode="markers", name="SELL",
            marker=dict(symbol="triangle-down", size=16, color="#ff1744", line=dict(width=1, color="black")),
            text=[f"SELL  score {s:.1f}<br>" + "<br>".join(r) for s, r in zip(sells["FinalScore"], sells["Reasons"])],
            hoverinfo="text+x",
        ))

    fig.update_layout(
        title=f"{symbol} — 5-Min Candles with Buy/Sell Signals (rule engine + ML)",
        xaxis_rangeslider_visible=False, template="plotly_dark", height=650,
    )
    fig.show()


## 13. Run it — fetch, train, backtest, and chart today's signals

In [ ]:
def run_pipeline(symbol=SYMBOL, interval=INTERVAL, period=PERIOD, plot=True, verbose=True):
    raw = fetch_data(symbol, interval, period)
    raw = add_indicators(raw)
    full = compute_signal_history(raw)
    full = add_forward_labels(full)

    train_idx, test_idx = chronological_split(full, FEATURE_COLS)
    model = train_ml_model(full, train_idx)
    full = add_ml_predictions(full, model)
    full = combine_rule_and_ml(full)
    full = add_arrow_markers(full)

    if verbose:
        print("=" * 64)
        print(f"{symbol} | fetched {len(full)} bars ({period} @ {interval})")
        print(f"Train period: {train_idx.min()}  ->  {train_idx.max()}  ({len(train_idx)} bars)")
        print(f"Test period:  {test_idx.min()}  ->  {test_idx.max()}  ({len(test_idx)} bars, out-of-sample)")
        print("=" * 64)
        print("ML model performance on the held-out test period:")
        y_test = full.loc[test_idx, "Label"]
        X_test = prep_features(full.loc[test_idx])
        mask = y_test.notna()
        print(classification_report(y_test[mask], model.predict(X_test[mask].values),
                                     target_names=["down", "flat", "up"], zero_division=0))

        trades = backtest_signals(full, test_idx)
        stats = summarize_backtest(trades)
        print_backtest_report(trades, stats, label="Combined rule+ML signal backtest (out-of-sample)")
        if plot:
            plot_equity_curve(stats, symbol)

    today = full.index[-1].date()
    day_df = full[full.index.date == today]

    last = full.iloc[-1]
    print(f"{symbol} | {full.index[-1]} | Last Close: {last['Close']:.2f}")
    print(f"SIGNAL: {last['FinalSignal']}  (rule score {last['Score']}, ML P(up)={last['P_up']:.2f}, "
          f"P(down)={last['P_down']:.2f}, final score {last['FinalScore']:.1f})")
    if last["Reasons"]:
        print("Rule-engine reasons:")
        for r in last["Reasons"]:
            print("  -", r)

    if plot:
        plot_signals_chart(day_df, symbol)

    return full, model


full_history, ml_model = run_pipeline()


## 14. Optional — auto-refresh loop for live intraday monitoring

In [ ]:
import time
from IPython.display import clear_output

def run_live(symbol=SYMBOL, interval=INTERVAL, period=PERIOD, refresh_seconds=300, iterations=100):
    """Re-fetches data, retrains the model on the latest window, recomputes
    signals, reruns the backtest, and redraws the chart every `refresh_seconds`
    (default 5 min), `iterations` times. Keep the Colab tab open for this to keep
    running; stop it any time with the cell's stop button."""
    for i in range(iterations):
        clear_output(wait=True)
        try:
            run_pipeline(symbol, interval, period, plot=True, verbose=True)
        except Exception as e:
            print("Error:", e)
        time.sleep(refresh_seconds)

# Uncomment to poll every 5 minutes for the rest of the session:
# run_live(refresh_seconds=300, iterations=100)
